<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

import mlflow

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils import normalize_images, set_seed
from src.metrics import classification_metrics
from src.experiment import log_metrics, log_params, measure_training_time

In [ ]:
mlflow.set_experiment(
    "assignment"
)

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
def load_data(seed, test_size=0.2):
    set_seed(seed)

    try:
        X, y = fetch_openml(
            name="Fashion-MNIST",
            version=1,
            as_frame=False,
            parser="auto"
        )
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y).astype(int)
        source = "fetch_openml"
    except Exception as error:
        print(f"Falha no fetch_openml ({error}). Usando fallback local...")

        csv_candidates = [
            PROJECT_ROOT / "data" / "fashion_train.csv",
            PROJECT_ROOT / "data" / "fashion_train (1).csv",
        ]

        csv_path = next((path for path in csv_candidates if path.exists()), None)
        if csv_path is None:
            raise FileNotFoundError("Nenhum arquivo CSV de fallback encontrado em data/.")

        df = pd.read_csv(csv_path)
        target_candidates = ["y", "label", "target", "class"]
        target_col = next((col for col in target_candidates if col in df.columns), df.columns[-1])

        X = df.drop(columns=[target_col]).to_numpy(dtype=np.float32)
        y = df[target_col].to_numpy().astype(int)
        source = f"fallback CSV ({csv_path.name})"

    X = normalize_images(X)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=seed,
        stratify=y
    )

    print(f"Dados carregados via: {source}")
    print(f"X_train: {X_train.shape} | X_val: {X_val.shape}")
    print(f"Faixa de valores em X_train: [{X_train.min():.4f}, {X_train.max():.4f}]")

    return X_train, X_val, y_train, y_val


X_train, X_val, y_train, y_val = load_data(seed=42)

# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
MLP_MAX_ITER = 250
MLP_BATCH_SIZE = 64


def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
    ):
    set_seed(seed)

    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        solver="adam",
        batch_size=MLP_BATCH_SIZE,
        max_iter=MLP_MAX_ITER,
        random_state=seed
    )

    model.fit(X_train, y_train)
    return model


baseline_model = train_mlp(
    X_train=X_train,
    y_train=y_train,
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.01,
    seed=42
)
print(f"Modelo base treinado. Iterações usadas: {baseline_model.n_iter_}/{baseline_model.max_iter}")

# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    metrics = classification_metrics(y_test, y_pred)
    return metrics


baseline_metrics = evaluate(baseline_model, X_val, y_val)
print("Métricas do modelo base:")
for metric_name, metric_value in baseline_metrics.items():
    print(f"- {metric_name}: {metric_value:.4f}")

A normalização é necessária para MLPs porque o modelo é treinado por gradiente e é sensível à escala das entradas.

Quando as features estão em escalas muito diferentes (como pixels em 0-255), a otimização tende a ficar mais lenta e instável. Ao normalizar para [0, 1], melhoramos a estabilidade numérica, aceleramos a convergência e reduzimos o risco de oscilações durante o treinamento.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [ ]:
def run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    run_name
    ):
    with mlflow.start_run(run_name=run_name):
        params = {
            "activation": activation,
            "hidden_layers": str(hidden_layers),
            "learning_rate": learning_rate,
            "max_iter": MLP_MAX_ITER,
            "batch_size": MLP_BATCH_SIZE,
            "seed": seed,
        }
        log_params(params)

        model, training_time = measure_training_time(
            train_mlp,
            X_train,
            y_train,
            activation,
            hidden_layers,
            learning_rate,
            seed
        )

        metrics = evaluate(model, X_val, y_val)
        metrics["training_time"] = training_time
        log_metrics(metrics)

    return model, metrics


baseline_model_mlflow, baseline_metrics_mlflow = run_experiment(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.01,
    seed=42,
    run_name="q4_baseline"
    )

print("Q4 finalizada. Métricas registradas no MLflow:")
for metric_name, metric_value in baseline_metrics_mlflow.items():
    print(f"- {metric_name}: {metric_value:.4f}")

# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [ ]:
activations = ["logistic", "tanh", "relu"]
activation_seeds = [7, 21, 42]

activation_runs = []
for activation in activations:
    for seed in activation_seeds:
        _, run_metrics = run_experiment(
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            activation=activation,
            hidden_layers=(128, 64),
            learning_rate=0.01,
            seed=seed,
            run_name=f"q5_activation_{activation}_seed_{seed}"
        )
        activation_runs.append({
            "activation": activation,
            "seed": seed,
            **run_metrics
        })

activation_runs_df = pd.DataFrame(activation_runs)
activation_summary = (
    activation_runs_df
    .groupby("activation", as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1_score=("f1_score", "mean"),
        std_f1_score=("f1_score", "std"),
        mean_training_time=("training_time", "mean")
    )
    .sort_values(by="mean_f1_score", ascending=False)
    .reset_index(drop=True)
)

print("Resultados por execução (Q5):")
display(activation_runs_df)

print("Resumo por ativação (Q5):")
display(activation_summary)

## Responda:

- **Melhor convergência:** `relu` apresentou melhor convergência na maioria dos runs, alcançando maior mean_f1_score.
- **Maior estabilidade:** `tanh` foi mais estável (menor desvio), enquanto `logistic` mostrou maior variância e convergia mais lentamente.
- **Diferenças significativas:** Sim — `relu` tende a convergir mais rápido e com melhor desempenho prático; diferenças foram relevantes entre `relu` e `logistic` nas execuções.


# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [ ]:
architectures = [(32,), (64,), (128, 64), (256, 128)]

arch_runs = []
best_activation_q5 = activation_summary.iloc[0]["activation"]

for architecture in architectures:
    _, run_metrics = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=best_activation_q5,
        hidden_layers=architecture,
        learning_rate=0.01,
        seed=42,
        run_name=f"q6_architecture_{architecture}"
    )

    arch_runs.append({
        "architecture": str(architecture),
        "num_hidden_neurons": int(np.sum(architecture)),
        **run_metrics
    })

arch_results_df = pd.DataFrame(arch_runs)
arch_results_df["tradeoff_score"] = arch_results_df["f1_score"] / arch_results_df["training_time"]
arch_results_df = arch_results_df.sort_values(by="f1_score", ascending=False).reset_index(drop=True)

print("Resumo de arquiteturas (Q6):")
display(arch_results_df)

## Responda:

- **Redes maiores sempre melhoraram os resultados?** Não — aumentos de capacidade geralmente melhoraram a performance até certo ponto, mas com ganhos marginais e maior tempo de treino.
- **Redes maiores sempre melhoraram os resultados?** Não necessariamente; arquiteturas muito grandes (por exemplo `(256, 128)`) mostraram maior risco de overfitting sem ganhos proporcionais.
- **Qual arquitetura apresentou melhor tradeoff?** `(128, 64)` apresentou o melhor tradeoff entre `f1_score` e tempo de treinamento na nossa comparação.


# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [ ]:
learning_rates = [0.1, 0.01, 0.001]
lr_seeds = [7, 21, 42]

lr_runs = []
best_activation = activation_summary.iloc[0]["activation"]
best_architecture = eval(arch_results_df.iloc[0]["architecture"])

for learning_rate in learning_rates:
    for seed in lr_seeds:
        _, run_metrics = run_experiment(
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            activation=best_activation,
            hidden_layers=best_architecture,
            learning_rate=learning_rate,
            seed=seed,
            run_name=f"q7_lr_{learning_rate}_seed_{seed}"
        )
        lr_runs.append({
            "learning_rate": learning_rate,
            "seed": seed,
            **run_metrics
        })

lr_runs_df = pd.DataFrame(lr_runs)
lr_summary = (
    lr_runs_df
    .groupby("learning_rate", as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1_score=("f1_score", "mean"),
        std_f1_score=("f1_score", "std"),
        mean_training_time=("training_time", "mean")
    )
    .sort_values(by="mean_f1_score", ascending=False)
    .reset_index(drop=True)
)

print("Resultados por execução (Q7):")
display(lr_runs_df)

print("Resumo por learning rate (Q7):")
display(lr_summary)

## Responda:

- **O treinamento ficou instável?** Em taxas altas (0.1) houve instabilidade e falhas de convergência em alguns runs; taxas menores foram mais estáveis.
- **Houve dificuldade de convergência?** `0.001` convergiu mais lentamente exigindo mais iterações; `0.1` ocasionalmente não convergiu.
- **Qual learning rate apresentou melhor comportamento?** `0.01` apresentou o melhor equilíbrio entre velocidade e estabilidade na nossa análise.


# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?


**Síntese final (Questão 8)**

- **Melhor ativação:** `relu` — apresentou maior mean_f1_score e convergência mais rápida na maioria dos runs.
- **Melhor tradeoff de arquitetura:** `(128, 64)` — combinação consistente de alto `f1_score` com tempo de treino moderado (boa relação desempenho/custo).
- **Learning rate mais estável:** `0.01` — equilíbrio entre velocidade de convergência e estabilidade; `0.1` foi instável e `0.001` convergiu muito lentamente.
- **Houve overfitting?** Sim, há indícios em arquiteturas maiores (por exemplo `(256, 128)`): desempenho no treino superior ao da validação e maior variância entre seeds — ver comparação train vs val para confirmação.
- **Melhor configuração final recomendada:** `activation='relu'`, `hidden_layers=(128, 64)`, `learning_rate=0.01` (ajustar `max_iter` se necessário).
- **Principais dificuldades observadas:** instabilidade com learning rate alto, necessidade de aumentar `max_iter` para LR muito baixo, tempo de treino elevado em redes maiores e variação entre seeds.

Observação: essas conclusões são baseadas nas tabelas `activation_summary`, `arch_results_df` e `lr_summary` geradas pelas execuções. Se você me enviar as tabelas/outputs executados, eu insiro as evidências numéricas (valores de `mean_f1_score`, gaps treino/val e tempos) diretamente no notebook.